# Phase 4 — Drift Detection + Early Warning
Multi-tool: **Python → R → Octave**

**Requires:** Phase 1 (snapshots.csv), Phase 2 (outputs/xai/), Phase 3 complete

In [ ]:
!pip install scipy psycopg2-binary sqlalchemy --quiet
print('Dependencies ready.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/CustomerHealthForensics'
import sys, os
sys.path.insert(0, f'{PROJECT_ROOT}/src')
sys.path.insert(0, f'{PROJECT_ROOT}/src/orchestration')

In [ ]:
# Upload drift source files
from google.colab import files
print('Upload: drift_engine.py, psi.py, ks_test.py (from previous phase34 build)')
print('Also: r/drift_validation.R, octave/psi_validation.m, octave/drift_math_validation.m')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f"{PROJECT_ROOT}/src/{fname}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, 'wb') as f: f.write(data)

In [ ]:
# Drift engine only (smoke test)
import pandas as pd, sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')
from drift_engine import DriftEngine

df_snap = pd.read_csv(f'{PROJECT_ROOT}/data/customers_snapshots.csv', nrows=240_000)
engine  = DriftEngine(df_snap, verbose=True)
report  = engine.run()

print(f"Severity: {report['overall_drift_severity']}")
print(f"Early warnings: {len(report['early_warnings'])}")
print(f"Retrain: {report['retraining_trigger']['model_retraining_required']}")

In [ ]:
# Full pipeline (all tools)
from orchestration.pipeline_runner import run_full_pipeline
from pathlib import Path

intel = run_full_pipeline(
    data_dir    = Path(f'{PROJECT_ROOT}/data'),
    output_dir  = Path(f'{PROJECT_ROOT}/outputs'),
    xai_dir     = Path(f'{PROJECT_ROOT}/outputs/xai'),
    skip_db     = True,    # set False if PostgreSQL available
    skip_r      = False,   # set True if R not installed
    skip_octave = False,   # set True if Octave not installed
    skip_segmentation = True,  # Phase 3 already done
)
import json; print(json.dumps({k:v for k,v in intel.items() if k not in ['segmentation']}, indent=2))

In [ ]:
# Inspect early warnings
import json
warnings = json.load(open(f'{PROJECT_ROOT}/outputs/drift/early_warnings.json'))
for w in warnings:
    xai = ' [XAI✓]' if w.get('xai_confirmed') else ''
    print(f"⚡ {w['feature']:30s}  psi={w['psi']:.3f}  {w['trend']:12s}  {w['velocity']}{xai}")

In [ ]:
# Octave PSI validation results
oct = json.load(open(f'{PROJECT_ROOT}/outputs/octave_outputs/psi_validation.json'))
print(f"Octave PSI agreement rate: {oct.get('agreement_rate',0):.1%}")
print(f"Validation passed: {oct.get('validation_passed',False)}")

In [ ]:
# Cross-tool retraining consensus
consensus = json.load(open(f'{PROJECT_ROOT}/outputs/retrain_consensus.json'))
print(consensus['interpretation'])
print(f"Python: {consensus['python_retrain']}  Octave: {consensus['octave_retrain']}")
print(f"Confidence: {consensus['confidence']}  ({consensus['n_tools_agree']}/{consensus['n_tools_checked']} agree)")

In [ ]:
# Plotly PSI chart
import plotly.express as px, pandas as pd
df_psi = pd.read_csv(f'{PROJECT_ROOT}/outputs/drift/psi_results.csv').sort_values('psi').tail(15)
color_map = {'stable':'green','monitor':'orange','significant_drift':'red'}
fig = px.bar(df_psi, x='psi', y='feature', orientation='h',
             color='psi_status', color_discrete_map=color_map,
             title='PSI by Feature (Python computed, Octave validated)')
fig.add_vline(x=0.1, line_dash='dash', line_color='orange')
fig.add_vline(x=0.2, line_dash='dash', line_color='red')
fig.show()

## ✅ Phase 4 Complete

```
outputs/drift/          ← Python drift engine
outputs/r_outputs/      ← R statistical validation
outputs/octave_outputs/ ← Octave mathematical validation
outputs/retrain_consensus.json ← cross-tool decision
outputs/phase34_intelligence.json ← feeds Phase 5
```